# Simplex Benchmark: Our RevisedSimplex vs HiGHS

Compare our RevisedSimplex against HiGHS on various LP problems.


In [ ]:
import sys, os, time, json
import numpy as np
import pandas as pd
import scipy.sparse as sp

sys.path.insert(0, "/data/dev/simplinho/dist")
import simplinho
import highspy
import pulp

np.random.seed(42)
print("simplinho:", simplinho.__version__, "git:", simplinho.__git_describe__)
print("HiGHS: v%d.%d.%d" % (highspy.HIGHS_VERSION_MAJOR, highspy.HIGHS_VERSION_MINOR, highspy.HIGHS_VERSION_PATCH))
print("numpy:", np.__version__)


## Solver Wrappers


In [ ]:
def solve_our(A, b, c, l, u, mode="auto", disable_presolve=False, sparse=False):
    opt = simplinho.RevisedSimplexOptions()
    opt.disable_presolve = disable_presolve
    opt.max_iters = 100000
    opt.verbose = 0
    opt.dualization = "off"
    opt.mode = getattr(simplinho.SimplexMode, mode.capitalize(), simplinho.SimplexMode.Auto)
    if sparse:
        opt.basis_sparse_backend = "sparse"
    simp = simplinho.RevisedSimplex(opt)
    t0 = time.perf_counter()
    res = simp.solve(A, b, c, l, u)
    t1 = time.perf_counter()
    tag = "dense" if not sparse else "sparse"
    if disable_presolve:
        tag += "_nopres"
    if mode != "auto":
        tag = mode + "_" + tag
    return {"name": tag, "status": res.status.name, "obj": res.obj if hasattr(res, "obj") else None, "x": res.x, "iters": res.iters, "time": t1 - t0, "error": None}

def solve_highs(A, b, c, l, u, method="dual"):
    m, n = A.shape
    lp = highspy.HighsLp()
    lp.num_col_ = n; lp.num_row_ = m
    lp.col_cost_ = c.tolist(); lp.col_lower_ = l.tolist(); lp.col_upper_ = u.tolist()
    lp.row_lower_ = b.tolist(); lp.row_upper_ = b.tolist()
    lp.sense_ = highspy.ObjSense.kMinimize; lp.offset_ = 0.0
    rows, vals = [], []
    for j in range(n):
        for i in range(m):
            if abs(A[i,j]) > 1e-15: rows.append(i); vals.append(float(A[i,j]))
    lp.a_matrix_ = highspy.HighsSparseMatrix()
    lp.a_matrix_.num_col_ = n; lp.a_matrix_.num_row_ = m
    lp.a_matrix_.index_ = rows; lp.a_matrix_.value_ = vals
    prev = 0; starts = []
    for j in range(n):
        starts.append(prev); prev += sum(1 for i in range(m) if abs(A[i,j]) > 1e-15)
    starts.append(len(rows))
    lp.a_matrix_.start_ = starts
    hs = highspy.Highs()
    hs.setOptionValue("log_to_console", "false")
    hs.setOptionValue("method", method)
    hs.passModel(lp)
    t0 = time.perf_counter()
    hs.run()
    t1 = time.perf_counter()
    status = hs.modelStatusToString(hs.getModelStatus())
    return {"name": method + "_highs", "status": status, "obj": hs.getObjectiveValue(), "x": hs.getSolution().col_value, "iters": hs.getInfo().simplex_iteration_count, "time": t1 - t0, "error": None}

print("Wrappers ready.")

## Problem Generators


In [ ]:
def gen_dense(m, n, density=0.3):
    A = np.random.randn(m, n)
    A[np.random.random((m, n)) > density] = 0.0
    c = np.random.randn(n)
    b = A @ np.random.randn(n)
    return A, b, c, -np.ones(n)*10.0, np.ones(n)*10.0

def gen_sparse(m, n, density=0.05):
    A = sp.random(m, n, density=density, format="csr", random_state=42, data_rvs=np.random.randn)
    c = np.random.randn(n)
    b = A @ np.random.randn(n)
    return A, b, c, -np.ones(n)*10.0, np.ones(n)*10.0

def gen_knapsack(m, n):
    A = np.random.exponential(1.0, (m, n))
    c = -np.random.exponential(1.0, n)
    b = np.array([np.sum(A[i]) * 0.5 for i in range(m)])
    return A, b, c, np.zeros(n), np.ones(n) * 2.0

def gen_transport(ns=5, nd=5):
    supply = np.random.exponential(10.0, ns)
    demand = np.random.exponential(10.0, nd)
    diff = supply.sum() - demand.sum()
    if diff > 0:
        demand[-1] += diff
    else:
        supply[-1] -= diff
    c = np.random.exponential(1.0, (ns, nd)).flatten()
    m = ns + nd; n_mat = ns * nd
    A = np.zeros((m, n_mat))
    for i in range(ns): A[i, i*nd:(i+1)*nd] = 1.0
    for j in range(nd): A[ns+j, j::ns] = 1.0
    return A, np.concatenate([supply, demand]), c, np.zeros(n_mat), np.ones(n_mat)*100.0

def gen_blended(m, n, density=0.3):
    A = np.random.randn(m, n)
    A[np.random.random((m, n)) > density] = 0.0
    c = np.random.randn(n)
    b = A @ np.random.randn(n)
    l = np.random.uniform(-5, -0.1, n)
    u = np.random.uniform(0.1, 5, n)
    return A, b, c, l, u

print("Generators ready.")


## Run Benchmarks

Smaller problem sizes for quick testing.


In [ ]:
problems = [
    ("dense_10x20", gen_dense, (10, 20, 0.3)),
    ("dense_20x50", gen_dense, (20, 50, 0.3)),
    ("dense_50x100", gen_dense, (50, 100, 0.3)),
    ("knapsack_20x50", gen_knapsack, (20, 50)),
    ("transport_3x3", gen_transport, (3, 3)),
    ("blended_20x50", gen_blended, (20, 50, 0.2)),
]

results = []
for name, gen_fn, args in problems:
    print("  " + name + "...", flush=True)
    A, b, c, l, u = gen_fn(*args)
    for kw in [
        {"name": "dense", "fn": lambda A=A, b=b, c=c, l=l, u=u: solve_our(A, b, c, l, u)},
        {"name": "dense_nopres", "fn": lambda A=A, b=b, c=c, l=l, u=u: solve_our(A, b, c, l, u, disable_presolve=True)},
        {"name": "dense_dual", "fn": lambda A=A, b=b, c=c, l=l, u=u: solve_our(A, b, c, l, u, mode="dual")},
        {"name": "highs_dual", "fn": lambda A=A, b=b, c=c, l=l, u=u: solve_highs(A, b, c, l, u, "dual")},
        {"name": "highs_auto", "fn": lambda A=A, b=b, c=c, l=l, u=u: solve_highs(A, b, c, l, u, "auto")},
    ]:
        try:
            r = kw["fn"]()
            r["problem"] = name
            results.append(r)
        except Exception as e:
            results.append({"problem": name, "name": kw["name"], "status": "ERROR", "obj": None, "iters": 0, "time": 0, "error": str(e)})
    print("    done (" + str(len(results)) + " total)")

print("Total: " + str(len(results)))

## Results


In [ ]:
df = pd.DataFrame(results)
cols = ["problem", "name", "status", "obj", "iters", "time"]
print(df[cols].to_string(index=False))


In [ ]:
pivot_t = df.pivot_table(index="problem", columns="name", values="time", aggfunc="mean")
print("TIME (seconds):")
print(pivot_t.round(5))


In [ ]:
pivot_o = df.pivot_table(index="problem", columns="name", values="obj", aggfunc="mean")
print("OBJECTIVE:")
print(pivot_o.round(6))


In [ ]:
pivot_s = df.pivot_table(index="problem", columns="name", values="status", aggfunc="first")
print("STATUS:")
print(pivot_s)


In [ ]:
pivot_i = df.pivot_table(index="problem", columns="name", values="iters", aggfunc="mean")
print("ITERATIONS:")
print(pivot_i.round(1))


In [ ]:
errs = df[df["status"] == "ERROR"][["problem", "name", "error"]]
if not errs.empty:
    display(errs)
else:
    print("No errors. All solvers succeeded.")
    h = df[(df["status"]=="Optimal") & (df["name"]=="dual_highs")].set_index("problem")["obj"]
    ours = df[(df["status"]=="Optimal") & (df["name"]=="dense")].set_index("problem")["obj"]
    common = h.index.intersection(ours.index)
    if len(common) > 0:
        re = (ours[common] - h[common]).abs() / h[common].abs()
        print("Objective rel error (our_dense vs highs_dual):")
        print("  max=" + repr(re.max()) + ", mean=" + repr(re.mean()))
        print("  within 1e-5: " + str((re < 1e-5).sum()) + "/" + str(len(re)))


## Warm Start Test


In [ ]:
A, b, c, l, u = gen_dense(50, 100, 0.2)
opt = simplinho.RevisedSimplexOptions()
opt.max_iters = 100000; opt.verbose = 0; opt.dualization = "off"
simp = simplinho.RevisedSimplex(opt)
t0 = time.perf_counter()
r1 = simp.solve(A, b, c, l, u)
print("Cold: %.4fs, iters=%d, obj=%.6f" % (time.perf_counter()-t0, r1.iters, r1.obj))
t0 = time.perf_counter()
r2 = simp.solve(A, b, c, l, u)
print("Warm: %.4fs, iters=%d, obj=%.6f" % (time.perf_counter()-t0, r2.iters, r2.obj))
simp2 = simplinho.RevisedSimplex(opt)
t0 = time.perf_counter()
r3 = simp2.solve(A, b, c, l, u)
print("Fresh: %.4fs, iters=%d, obj=%.6f" % (time.perf_counter()-t0, r3.iters, r3.obj))


## Presolve Impact


In [ ]:
A, b, c, l, u = gen_dense(50, 100, 0.3)
opt = simplinho.RevisedSimplexOptions()
opt.max_iters = 100000; opt.verbose = 0; opt.dualization = "off"

opt.disable_presolve = False
simp = simplinho.RevisedSimplex(opt)
t0 = time.perf_counter()
r_on = simp.solve(A, b, c, l, u)
t1 = time.perf_counter()
print("With presolve: %.4fs, iters=%d, obj=%.6f" % (t1-t0, r_on.iters, r_on.obj))

opt.disable_presolve = True
simp2 = simplinho.RevisedSimplex(opt)
t0 = time.perf_counter()
r_off = simp2.solve(A, b, c, l, u)
t1 = time.perf_counter()
print("Without presolve: %.4fs, iters=%d, obj=%.6f" % (t1-t0, r_off.iters, r_off.obj))


In [ ]:
df.to_csv("/data/dev/simplinho/benchmark_results.csv", index=False)
print("Saved " + str(len(df)) + " results.")
